In [ ]:

import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box
from scipy.spatial.distance import pdist,squareform
import seaborn as sns
import pytim
import warnings
import pandas as pd


In [ ]:
### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list
    
    
    
def surface_comp2(u, start=4000, stop=-1, step=10000):
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.0,
            max_layers=3,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        mid_resids = np.unique(layer2_atoms.resids)
        inner_resids = np.unique(layer3_atoms.resids)
        
        
        return inner_resids ,mid_resids,outer_resids,res_list
    

def surface_comp3(u, start=4000, stop=-1, step=10000):
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.2,
            max_layers=2,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        #layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        inner_resids = np.unique(layer2_atoms.resids)
        
        
        return inner_resids,outer_resids,res_list
    

    


def visu3(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('surface',sele='protein',opacity=0.8,color='green')
    view.add_representation('ball+stick',sele='nucleic',aspectRatio=2)
    view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    view.add_representation('surface',sele='nucleic',opacity=0.8)
    view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    #view.add_representation('ball+stick',sele='water',aspectRatio=2)
    view.add_unitcell(color='white')
    return view


def visul5(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=2,
                            color='green')

    view.add_unitcell(color='white')
    return view


def visul6(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(SOL ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=8,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view


def visul7(unive,s1,s2,s3):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=1.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"(protein) and ({s3})",
                            opacity=1.5,
                            aspectRatio=4,
                            color='red')
    
    

    view.add_unitcell(color='white')
    return view


def visul8(unive,s1,s2,s3,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"(protein) and ({s3})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='red')
    
    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=4,
                            color='orange')
        

    view.add_unitcell(color='white')
    return view


def visul9(unive,s1,s2,s3,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('surface',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('surface',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('surface',sele=f"(protein) and ({s3})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='red')
    
    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=4,
                            color='orange')
        

    view.add_unitcell(color='white')
    return view



def visul10(unive,s1,s2,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='orange')
        

    view.add_unitcell(color='white')
    return view




def visul11(unive,s1,s2,water1,water2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
#     view.add_representation('surface',sele=f"(nucleic) and ( {s1})",
#                             opacity=0.5,aspectRatio=2,color='red')  
    
    
#     view.add_representation('surface',sele=f"(protein) and ( {s1})",
#                             opacity=0.5,aspectRatio=4,color='yellow')  
    

#     view.add_representation('surface',sele=f"(protein) and ({s2})",
#                             opacity=0.5,
#                             aspectRatio=4,
#                             color='green')

#     view.add_representation('surface',sele=f"(protein) and ({s3})",
#                             opacity=0.5,
#                             aspectRatio=4,
#                             color='red')

    view.add_representation('ball+stick',sele=f"{water1}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='blue')

    
    view.add_representation('ball+stick',sele=f"{water2}",
                            opacity=1.5,
                            aspectRatio=6,
                            color='red')
        

    view.add_unitcell(color='white')
    return view


In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/polar//output.top',
                       xyz='../gxg_comps/3_res/polar//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 #'../gxg_comps/3_res/polar/traj_2-nw.dcd',
                 '../gxg_comps/3_res/polar/trajectory.dcd'
                )
univ.trajectory


In [ ]:

view = show_str(univ)
view

In [ ]:

view.render_image(factor=5,
                  transparent=True,
                 trim=True)

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/nonpolar//output.top',
                       xyz='../gxg_comps/3_res/nonpolar//output.gro')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 #'../gxg_comps/3_res/nonpolar//trajectory.dcd',
                '../gxg_comps/3_res/nonpolar/traj_2-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/nonpolar//output.top',
                       xyz='../gxg_comps/9_res/nonpolar//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/nonpolar/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/charged//output.top',
                       xyz='../gxg_comps/9_res/charged/output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/charged/traj.dcd',
                 #'../gxg_comps/9_res/charged/traj-nw.dcd'
                )
univ.trajectory


In [ ]:
show_str(univ)

In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal/3_spread//output.top',
                       xyz='../gxg_comps/9_res/equal/3_spread//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/3_spread/traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal/3_close///output.top',
                       xyz='../gxg_comps/9_res/equal/3_close///output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/3_close//traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal/2_close///output.top',
                       xyz='../gxg_comps/9_res/equal/2_close///output.gro')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 #'../gxg_comps/9_res/equal/2_close//traj.dcd',
                 '../gxg_comps/9_res/equal/2_close/traj-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal/2_spread//output.top',
                       xyz='../gxg_comps/9_res/equal/2_spread//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/2_spread//traj.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/nonpolar///output.top',
                       xyz='../gxg_comps/3_res/nonpolar//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/nonpolar/trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory


In [ ]:

inner_layers,mid_layers,outer_layers,res_list = surface_comp2(univ, 
                                                              start=110000,
                                                              stop=110001, 
                                                              step=1)


In [ ]:

inner_layer_list = ' '
for item in inner_layers:
    inner_layer_list += str(item) + ' '
    
    
outer_layer_list = ' '
for item in outer_layers:
    outer_layer_list += str(item) + ' '
    
mid_layer_list = ' '
for item in mid_layers:
    mid_layer_list += str(item) + ' '
    
residue_list = ' '
for item in res_list:
    residue_list += str(item) + ' '
    

In [ ]:
inner_layer_list

In [ ]:

dist_water = 3.0

univ.trajectory[110000]

# Step 2: Water within dist_water of the CORE (clustered) residues

select = f" name OW HW1 HW2 and (around {dist_water} (protein and resid {inner_layer_list}))"

water_near_core = univ.select_atoms(select)

water_resids = water_near_core.indices
    
idx_sel = "@" + ",".join(str(r) for r in water_resids)


In [ ]:
visul7(univ,inner_layer_list,mid_layer_list,outer_layer_list)


In [ ]:

visul9(univ,inner_layer_list,mid_layer_list,outer_layer_list,idx_sel)


### Two divisons

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/polar//output.top',
                       xyz='../gxg_comps/3_res/polar//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/polar/trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

univ.trajectory


In [ ]:


stru = parmed.load_file('../gxg_comps/9_res/equal/2_spread//output.top',
                       xyz='../gxg_comps/9_res/equal/2_spread//output.gro')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/2_spread/traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory



In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/nonpolar///output.top',
                       xyz='../gxg_comps/3_res/nonpolar//output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/nonpolar/trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
univ.trajectory

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/charged/output.top',
                       xyz='../gxg_comps/3_res/charged/output.gro')

# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/charged/trajectory.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

univ.trajectory


In [ ]:


inner_layers,outer_layers,res_list = surface_comp3(univ, 
                                                    start=100000,
                                                    stop=100001, 
                                                    step=1)


In [ ]:
all_list = ' '
for item in res_list:
    all_list += str(item) + ' '


inner_layer_list = ' '
for item in inner_layers:
    inner_layer_list += str(item) + ' '
    
    
outer_layer_list = ' '
for item in outer_layers:
    outer_layer_list += str(item) + ' '
    

In [ ]:

dist_water_inner = 3.3
dist_water_outer = 3.0

univ.trajectory[100000]

# Step 2: Water within dist_water of the CORE (clustered) residues

select_inner = f" name OW  and (around {dist_water_inner} (protein and resid {inner_layer_list}))"

select_outer = f" name OW  and (around {dist_water_outer} (protein and resid {outer_layer_list}))"

water_inner = univ.select_atoms(select_inner)

water_outer = univ.select_atoms(select_outer)

water_inner_resids = water_inner.indices
water_outer_resids = water_outer.indices

common_resids = set(water_inner_resids).intersection(set(water_outer_resids))

water_inner_resids = set(water_inner_resids)-common_resids
    
idx_sel = "@" + ",".join(str(r) for r in water_inner_resids)
idx_sel_2 = "@" + ",".join(str(r) for r in water_outer_resids)


In [ ]:
len(set(water_inner_resids)), len( set(water_outer_resids)), len( set(common_resids)),
len(inner_layers),len(outer_layers),len(res_list)


In [ ]:

#visul10(univ,inner_layer_list,outer_layer_list,idx_sel)

visul10(univ,all_list,outer_layers,idx_sel)


In [ ]:

# select atoms (example)
sel = univ.select_atoms("all")

# go to a specific frame (e.g., frame 1000)
univ.trajectory[100000]

# write PDB
sel.write("frame_100000.pdb")


In [ ]:

u = mda.Universe(stru,'frame_100000.pdb')


In [ ]:

solute = u.select_atoms(f"(protein and resid {inner_layer_list})")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(mda.transformations.unwrap(u.select_atoms("all") ),
    center_in_box(solute),
    wrap(u.select_atoms("all"), compound='fragments')
)




In [ ]:

idx_sel =  " ".join(str(r) for r in water_inner_resids)
idx_sel_2 =  " ".join(str(r) for r in water_outer_resids)

sel = solute[solute.positions[:, 0] > 50]
sel = "  ".join(str(r) for r in sel.resids)


outer = u.select_atoms(f"(protein and resid {outer_layer_list})")
outer_sel = outer[outer.positions[:, 0] > 50]
outer_sel = "  ".join(str(r) for r in outer_sel.resids)

inner_w = u.select_atoms(f"(index {idx_sel})")
inner_w_sel = inner_w[inner_w.positions[:, 0] > 50]
inner_w_sel = "@" + ",".join(str(r) for r in inner_w_sel.indices)

outer_w = u.select_atoms(f"(index {idx_sel_2})")
outer_w_sel = outer_w[outer_w.positions[:, 0] > 50]
outer_w_sel = "@" + ",".join(str(r) for r in outer_w_sel.indices)



In [ ]:
visul10(u,sel,outer_sel,inner_w_sel)


In [ ]:

visul11(u,sel,outer_sel,inner_w_sel,outer_w_sel)


### Normal Visulzation 

In [ ]:
import pandas as pd
import nglview as nv
import MDAnalysis as mda
import parmed

import nglview as nv
import MDAnalysis as mda
from MDAnalysis.transformations import wrap, center_in_box

def show_str(u):
    """
    Show MDAnalysis Universe in nglview with each amino acid 
    type colored differently.
    """
    view = nv.show_mdanalysis(u)

    # Define colors for amino acids (extend as needed)
    aa_colors = {
        "GLY":"silver" ,
        "ALA": "gray",
        "VAL": "lime",
        "LEU": "green",
        "ILE": "forestgreen",
        "SER": "cyan",
        "THR": "teal",
        "ASN": "blue",
        "GLN": "royalblue",
        "ASP": "red",
        "GLU": "orangered",
        "LYS": "magenta",
        "ARG": "purple",
        "HIS": "gold",
        "PHE": "orange",
        "TYR": "darkorange",
        "TRP": "brown",
        "CYS": "yellow",
        "MET": "khaki",
        "PRO": "pink",
    }

    # Add a representation for each residue type
    for res, col in aa_colors.items():
        view.add_representation(
            'ball+stick',
            sele=f"{res}",
            color=col,
            aspectRatio=5
        )

    # Show the simulation box
    view.add_unitcell()
    
    return view



def show_str(u):
    """
    Show MDAnalysis Universe in nglview with amino acids 
    colored by chemical property group.
    """
    view = nv.show_mdanalysis(u)

    # Define groups of amino acids
    aa_groups = {
        "Polar": {
            "resnames": ["SER", "THR", "ASN", "GLN", "CYS"],
            "color": "cyan",
        },
        "Positive": {
            "resnames": ["LYS", "ARG"],
            "color": "blue",
        },
        "Negative": {
            "resnames": ["ASP", "GLU"],
            "color": "red",
        },
        "Aromatic": {
            "resnames": ["PHE", "TYR", "TRP", "HIS"],
            "color": "orange",
        },
        "Hydrophobic": {
            "resnames": ["ALA", "VAL", "LEU", "ILE", "MET", "PRO"],
            "color": "green",
        },
        "spac": {
            "resnames": ["GLY"],
            "color": "silver",
        },
        
    }
    # Add representations by group
    for group, info in aa_groups.items():
        sele_str = " or ".join([f" {r}" for r in info["resnames"]])
        view.add_representation(
            "ball+stick",
            sele=sele_str,
            color=info["color"],
            aspectRatio=5
        )


        
    view.add_representation(
            "ball+stick",
            sele='all',
            color=info["color"],
            aspectRatio=1,
        opacity=0.3
        )
    

    # Show the simulation box
    view.add_unitcell()

    return view


In [ ]:
aa_groups = {
    "Polar": {
        "resnames": ["SER", "THR", "ASN", "GLN", "CYS"],
        "color": "cyan",
    },
    "Positive": {
        "resnames": ["LYS", "ARG", "HIS"],
        "color": "blue",
    },
    "Negative": {
        "resnames": ["ASP", "GLU"],
        "color": "red",
    },
    "Aromatic": {
        "resnames": ["PHE", "TYR", "TRP"],
        "color": "orange",
    },
    "Hydrophobic": {
        "resnames": ["ALA", "VAL", "LEU", "ILE", "MET", "PRO"],
        "color": "green",
    },
    "spac": {
        "resnames": ["GLY"],
        "color": "silver",
    },

}

for group, info in aa_groups.items():
    sele_str = " ".join([f" {r}" for r in info["resnames"]])
    print(sele_str)

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/aromatic/output-nw.gro')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/aromatic/trajectory-nw.dcd',
                 refresh_offsets=True)

In [ ]:

show_str(u)

In [ ]:

stru = parmed.load_file('../gxg_comps/3_res/nonpolar//output-nw.gro')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/nonpolar/traj_2-nw.dcd',
                 refresh_offsets=True)

In [ ]:

show_str(u)

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/equal/output-nw.gro')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/traj-nw.dcd',
                 refresh_offsets=True)


In [ ]:
#from MDAnalysis.transformations import nojump

ag = u.atoms
solute = u.select_atoms("protein") 
transform = mda.transformations.unwrap(ag.atoms)
u.trajectory.add_transformations(transform,
                                 #center_in_box(solute)
                                )

In [ ]:

show_str(u)


In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/polar/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/polar/traj_2-nw.dcd')

u.trajectory

In [ ]:
from MDAnalysis import transformations as trans
a        = u.atoms
workflow = [trans.wrap(a), trans.unwrap(a)]
u.trajectory.add_transformations(*workflow)

In [ ]:

solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(mda.transformations.unwrap(u.select_atoms("all") ),
    center_in_box(solute),
    #wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/charged//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 #'../gxg_comps/3_res/charged/trajectory-nw.dcd',
                 '../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )
u.trajectory

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/aromatic/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/aromatic//trajectory-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/polar//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/polar//trajectory-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


###  9 reidues

In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/charged//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/charged//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/aromatic//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/aromatic//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )



In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/polar//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/polar//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/nonpolar//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/nonpolar//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )


In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal/2_close/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/2_close//traj-nw.dcd',
                )

u.trajectory

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal/2_spread//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/2_spread//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal/3_close//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/3_close//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


In [ ]:
stru = parmed.load_file('../gxg_comps/9_res/equal/3_spread//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/9_res/equal/3_spread//traj-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = u.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
u.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:

view = show_str(u)
view


#### checking cluster on DBscan

In [ ]:
import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np

### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list

def visu3(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('surface',sele='protein',opacity=0.8,color='green')
    view.add_representation('ball+stick',sele='nucleic',aspectRatio=2)
    view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    view.add_representation('surface',sele='nucleic',opacity=0.8)
    view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    #view.add_representation('ball+stick',sele='water',aspectRatio=2)
    view.add_unitcell(color='white')
    return view

def visul5(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=2,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/equal/output.top',
                       xyz='../gxg_comps/3_res/equal/output.gro')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/equal/traj_2-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/aromatic//output.top',
                       xyz='../gxg_comps/3_res/aromatic//output.gro')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
univ = mda.Universe(stru,
                 '../gxg_comps/3_res/aromatic/trajectory-nw.dcd',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )

In [ ]:
solute = univ.select_atoms("protein")   # or "resname RNA" or "not resname SOL"

# Define transformations:
# 1. Center solute in box
# 2. Wrap everything around the solute
univ.trajectory.add_transformations(
    center_in_box(solute, wrap=True),
    wrap(solute, compound='fragments')
)

In [ ]:
univ.trajectory

In [ ]:
u=univ
for ts in u.trajectory[44000:44001]:
        
        centroid, to_cluster, clustered_list, res_list = clustering(u, ts,eps=9)
         
        selection = 'resindex '
        for item in res_list:
                selection += str(item)+ ' or resindex '
        selection = selection[0:-13]
        
        res_list = np.array(res_list)
        
        selection_2 = ' '
        for item in res_list:
                selection_2 += str(item)+ ' '       

        sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)
        
        selection_3 = ' '
        for item in sel:
                selection_3 += str(item)+ ' '

In [ ]:
visul5(univ,selection_2,selection_3)

In [ ]:
len(res_list)